<a href="https://colab.research.google.com/github/KurtGabrielAnduque/CPE-312-BSCPE31S3/blob/main/FINAL_COLAB_GROUP6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<font size = 6><b> Loading</b> and <b>Extraction</b> of features from each audio

In [ ]:
# run the google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install noisereduce

In [ ]:
# import the necessary libraries for the extraction of information from each wav
import os
import librosa
import noisereduce as nr
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
# prepare the file path of each gender voices from kaggle
file_path = {
    "male" : "/content/drive/MyDrive/BSCPE31S3-CPE312/VOICES/VOICES/data/male",
    "female" : "/content/drive/MyDrive/BSCPE31S3-CPE312/VOICES/VOICES/data/female"
}

In [ ]:
# check the number of data in each file path
for gender, gender_files in file_path.items():
  counter = 0
  for wav in os.listdir(gender_files):
    counter += 1
  print(f'{gender} : {counter} files')

male : 10379 files
female : 5768 files


In [ ]:
def preprocess_audio(file_path, sr=22050):

    # each sample is around 3 seconds long in this dataset
    y, sr = librosa.load(file_path, sr=sr)

    # Noise reduction using first 0.2s as noise sample
    # Reduces background noise such as hums, hisses, or ambient sounds using filters or noise reduction algorithms.
    noisy_part = y[:sr//5]
    y = nr.reduce_noise(y=y, sr=sr, y_noise=noisy_part, stationary=True)

    # Trim silence
    # Removes unnecessary silence from the beginning and end of the audio.
    y, _ = librosa.effects.trim(y, top_db=20)

    # Normalize
    # Ensures all audio signals are on the same volume scale by scaling the waveform so its peak is consistent across samples.
    y = librosa.util.normalize(y)

    return y, sr

In [ ]:
def extract_features(y, sr):
    features = {}

    # MFCCs (13 coefficients + deltas)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    features.update({f'mfcc_{i}': np.mean(v) for i,v in enumerate(mfcc)})

    # Spectral Features
    # centroid mean
    # where the "center of mass" of the sound frequencies is
    features['spectral_centroid'] = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))

    # speactra bandwidth mean
    features['spectral_bandwidth'] = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    features['spectral_contrast'] = np.mean(librosa.feature.spectral_contrast(y=y, sr=sr))

    # rolloff
    # use for the indication of the “brightness” of the sound.
    features['spectral_rolloff'] = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))

    # Pitch and ZCR
    features['zcr'] = np.mean(librosa.feature.zero_crossing_rate(y))

    f0 = librosa.yin(y, fmin=30, fmax=400)
    features['pitch_mean'] = np.mean(f0[f0 > 0])

    return features

In [ ]:
def create_dataset(gender_paths, sample_limit=None):

    rows = []

    for gender, path in gender_paths.items():
        files = [f for f in os.listdir(path) if f.endswith('.wav')]
        if sample_limit:
            files = files[:sample_limit]

        for filename in tqdm(files, desc=f'Processing {gender}'):
            try:
                # Preprocessing
                y, sr = preprocess_audio(os.path.join(path, filename))

                # Feature extraction
                features = extract_features(y, sr)
                features['gender'] = gender

                rows.append(features)
            except Exception as e:
                print(f"Error processing {filename}: {str(e)}")

    return pd.DataFrame(rows)

In [ ]:
# CREATE THE DF frame I balance the data between female and male
# 4000 each gender to avoid bias
df = create_dataset(file_path, sample_limit=4000)
# SAVE THE CREATED DATAFRAME AS CSV before moving to cleaning phase
df.to_csv('gender_voice_features_NoLimit_samples.csv', index=False)

Processing female: 100%|██████████| 4000/4000 [14:20<00:00,  4.65it/s]


In [ ]:
# check the contents of the dataframe
df.head()

,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,spectral_centroid,spectral_bandwidth,spectral_contrast,spectral_rolloff,zcr,pitch_mean,gender
0,-219.602371,97.603912,-54.517742,46.531658,-27.272753,20.433603,-29.672255,10.032966,-15.942534,-2.571034,-8.128550,-12.223826,12.524948,2329.547856,1444.909573,25.781375,3781.277622,0.160898,113.628085,male
1,-232.477356,141.070114,-32.062172,46.708454,-1.240481,-2.158743,-8.697986,3.969032,-11.782547,2.448183,-2.235656,-6.379191,2.610514,1590.933317,1264.998614,23.929932,2758.687721,0.091041,128.639345,male
2,-290.651337,139.408981,-25.245720,45.020210,-2.888519,-1.860083,-4.964668,2.754945,-15.611351,1.659974,-2.835146,-5.640999,3.086393,1515.115904,1322.423438,23.864699,2745.423584,0.085490,127.646279,male
3,-207.755615,152.285233,-39.265598,45.407623,7.981366,2.951322,-5.880070,0.456000,-18.027618,1.799478,-2.355123,-4.659540,3.568814,1441.507622,1250.887235,23.600023,2595.717210,0.077342,117.808694,male
4,-317.883179,80.067551,-17.273602,71.706085,17.568960,-15.172716,-33.383091,-0.431796,-15.974643,-17.893518,0.099104,-13.188399,1.710593,1849.633890,1169.948353,31.510350,2987.261548,0.127228,131.673358,male


<font size =6>CLEANING THE DATASET

In [ ]:
# IMPORT THE NECESSARY LIBRARIES FOR CLEANING OF THE DATASET
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
# import the dataset from the recent created csv
df = pd.read_csv("/content/gender_voice_features.csv")
df.head()

,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,spectral_centroid,spectral_bandwidth,spectral_contrast,spectral_rolloff,zcr,pitch_mean,gender
0,-219.60237,97.60391,-54.517742,46.531660,-27.272753,20.433603,-29.672255,10.032966,-15.942534,-2.571034,-8.128550,-12.223826,12.524948,2329.547856,1444.909573,25.781375,3781.277622,0.160898,113.628085,male
1,-232.47736,141.07011,-32.062172,46.708454,-1.240481,-2.158743,-8.697986,3.969032,-11.782547,2.448183,-2.235656,-6.379191,2.610514,1590.933317,1264.998614,23.929932,2758.687721,0.091041,128.639345,male
2,-290.65134,139.40898,-25.245720,45.020210,-2.888519,-1.860083,-4.964668,2.754945,-15.611351,1.659974,-2.835146,-5.640999,3.086393,1515.115904,1322.423438,23.864699,2745.423584,0.085490,127.646279,male
3,-207.75562,152.28523,-39.265600,45.407623,7.981366,2.951322,-5.880070,0.456000,-18.027618,1.799478,-2.355123,-4.659540,3.568814,1441.507622,1250.887235,23.600023,2595.717210,0.077342,117.808694,male
4,-317.88318,80.06755,-17.273602,71.706085,17.568960,-15.172716,-33.383090,-0.431796,-15.974643,-17.893518,0.099104,-13.188399,1.710594,1849.633890,1169.948353,31.510350,2987.261548,0.127228,131.673358,male


In [ ]:
# check the data types of each columns
df.dtypes

,0
mfcc_0,float64
mfcc_1,float64
mfcc_2,float64
mfcc_3,float64
mfcc_4,float64
mfcc_5,float64
mfcc_6,float64
mfcc_7,float64
mfcc_8,float64
mfcc_9,float64


In [ ]:
# change the data type of gender from object into string data type
df['gender'] = df['gender'].astype("string")

In [ ]:
# check the dtype again
df.dtypes

,0
mfcc_0,float64
mfcc_1,float64
mfcc_2,float64
mfcc_3,float64
mfcc_4,float64
mfcc_5,float64
mfcc_6,float64
mfcc_7,float64
mfcc_8,float64
mfcc_9,float64


In [ ]:
# check for null values
df.isnull().sum()

,0
mfcc_0,0
mfcc_1,0
mfcc_2,0
mfcc_3,0
mfcc_4,0
mfcc_5,0
mfcc_6,0
mfcc_7,0
mfcc_8,0
mfcc_9,0


In [ ]:
# check for duplicates
df.duplicated().sum()

# drop duplicates
df_new = df.drop_duplicates()
print(df_new.duplicated().sum())

0


In [ ]:
# reset the index of the daaset
df_new = df_new.reset_index(drop = True)

In [ ]:
# recheck the dataset again
df_new.head()

,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,spectral_centroid,spectral_bandwidth,spectral_contrast,spectral_rolloff,zcr,pitch_mean,gender
0,-219.60237,97.60391,-54.517742,46.531660,-27.272753,20.433603,-29.672255,10.032966,-15.942534,-2.571034,-8.128550,-12.223826,12.524948,2329.547856,1444.909573,25.781375,3781.277622,0.160898,113.628085,male
1,-232.47736,141.07011,-32.062172,46.708454,-1.240481,-2.158743,-8.697986,3.969032,-11.782547,2.448183,-2.235656,-6.379191,2.610514,1590.933317,1264.998614,23.929932,2758.687721,0.091041,128.639345,male
2,-290.65134,139.40898,-25.245720,45.020210,-2.888519,-1.860083,-4.964668,2.754945,-15.611351,1.659974,-2.835146,-5.640999,3.086393,1515.115904,1322.423438,23.864699,2745.423584,0.085490,127.646279,male
3,-207.75562,152.28523,-39.265600,45.407623,7.981366,2.951322,-5.880070,0.456000,-18.027618,1.799478,-2.355123,-4.659540,3.568814,1441.507622,1250.887235,23.600023,2595.717210,0.077342,117.808694,male
4,-317.88318,80.06755,-17.273602,71.706085,17.568960,-15.172716,-33.383090,-0.431796,-15.974643,-17.893518,0.099104,-13.188399,1.710594,1849.633890,1169.948353,31.510350,2987.261548,0.127228,131.673358,male


In [ ]:
df_new['gender'].unique()

<StringArray>
['male', 'female']
Length: 2, dtype: string

In [ ]:
# since we are going to use the logistic regression for classification
# I think it is better if we turn the gender into binary
# female = 1
# male = 0
df_new["gender"] = df_new["gender"].apply(lambda x: 1 if x == "female" else 0)

# check the unique values to verify
df_new['gender'].unique()

array([0, 1])

In [ ]:
df_new.head()

,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,spectral_centroid,spectral_bandwidth,spectral_contrast,spectral_rolloff,zcr,pitch_mean,gender
0,-219.60237,97.60391,-54.517742,46.531660,-27.272753,20.433603,-29.672255,10.032966,-15.942534,-2.571034,-8.128550,-12.223826,12.524948,2329.547856,1444.909573,25.781375,3781.277622,0.160898,113.628085,0
1,-232.47736,141.07011,-32.062172,46.708454,-1.240481,-2.158743,-8.697986,3.969032,-11.782547,2.448183,-2.235656,-6.379191,2.610514,1590.933317,1264.998614,23.929932,2758.687721,0.091041,128.639345,0
2,-290.65134,139.40898,-25.245720,45.020210,-2.888519,-1.860083,-4.964668,2.754945,-15.611351,1.659974,-2.835146,-5.640999,3.086393,1515.115904,1322.423438,23.864699,2745.423584,0.085490,127.646279,0
3,-207.75562,152.28523,-39.265600,45.407623,7.981366,2.951322,-5.880070,0.456000,-18.027618,1.799478,-2.355123,-4.659540,3.568814,1441.507622,1250.887235,23.600023,2595.717210,0.077342,117.808694,0
4,-317.88318,80.06755,-17.273602,71.706085,17.568960,-15.172716,-33.383090,-0.431796,-15.974643,-17.893518,0.099104,-13.188399,1.710594,1849.633890,1169.948353,31.510350,2987.261548,0.127228,131.673358,0


<font size = 6> CREATING MACHINE LEARNING MODELS

# <font size= 5><b>LOGISTIC REGRESSION

In [ ]:
# seperate the features into target columns
X = df_new.drop(columns = "gender")
Y = df_new['gender']

In [ ]:
# IMPORT THE NECESSARY LIBRARIES FOR ML MODELS
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# create training set for each

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size =0.3, random_state=42)


# verify the shapes of each sets

print(f'train x: {x_train.shape}')
print(f'trian y: {y_train.shape}')
print(f'test x: {x_test.shape}')
print(f'train y: {y_test.shape}')

train x: (4839, 19)
trian y: (4839,)
test x: (2074, 19)
train y: (2074,)


In [ ]:
# creat the logistic regression model
model = LogisticRegression()
# train the model with the provided training set
model.fit(x_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [ ]:
# after training the model
# lets try predicting into unseen values
y_pred = model.predict(x_test)

In [ ]:
# using the classification_report we can check the accuracy of the model

print(classification_report(y_pred,y_test))

              precision    recall  f1-score   support

           0       0.97      0.97      0.97      1064
           1       0.97      0.97      0.97      1010

    accuracy                           0.97      2074
   macro avg       0.97      0.97      0.97      2074
weighted avg       0.97      0.97      0.97      2074



<font size =6> THE MODEL WAS ABLE TO GENERATE A 97% ON EACH CLASSIFICATION REPORT

<b> LETS TRY WITH CONFUSION MATRIX

In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[1035   35]
 [  29  975]]


In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, x_train, y_train, cv=5)
print("Cross-validation scores:", scores)
print("Mean CV accuracy:", scores.mean())

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Cross-validation scores: [0.97417355 0.97004132 0.96900826 0.97004132 0.97828335]
Mean CV accuracy: 0.9723095626757374


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

<font size= 5>IT SHOWS A 97% in every classification report ItS better if we test it with real dataset from our group

In [ ]:
#create a separation file for male and female
file_group6 = {
    "male" : "/content/drive/MyDrive/BSCPE31S3-CPE312/GROUP6_VOICE_TEST_SET/male",
    "female" : "/content/drive/MyDrive/BSCPE31S3-CPE312/GROUP6_VOICE_TEST_SET/female"
}

In [ ]:
# extract the features in each wav files
df = create_dataset(file_group6, sample_limit=None)
# SAVE THE CREATED DATAFRAME AS CSV before moving to cleaning phase
df.to_csv('gender_voice_features_GROUP6.csv', index=False)

Processing male:   0%|          | 0/12 [00:00<?, ?it/s]/tmp/ipython-input-4266735911.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
Processing male:  42%|████▏     | 5/12 [00:00<00:00,  9.17it/s]/tmp/ipython-input-4266735911.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
Processing male:  50%|█████     | 6/12 [00:00<00:00,  7.06it/s]/tmp/ipyt

In [ ]:
# clean the file just like the recent
group6 = pd.read_csv("/content/gender_voice_features_GROUP6.csv")
group6.head()

,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,spectral_centroid,spectral_bandwidth,spectral_contrast,spectral_rolloff,zcr,pitch_mean,gender
0,-277.84058,153.75758,5.474815,-15.882857,-68.344620,-69.797670,-5.401819,5.932216,-25.657804,-30.918259,-28.823265,-11.045218,0.786485,963.045077,926.665048,39.428446,1284.058902,0.057900,115.878161,male
1,-296.20380,187.58519,62.128387,14.630950,-19.518356,36.246716,10.381476,5.048954,-2.127603,-22.404743,-0.736466,-4.043631,-6.394054,599.930886,942.011176,29.528319,1046.700917,0.027047,118.972232,male
2,-293.13947,195.18689,59.171772,1.359976,-22.836230,-10.960691,-17.511960,-11.966806,-9.834209,-25.713228,-26.245882,-12.430363,-11.446730,656.439295,657.650559,36.625321,1053.108215,0.040115,131.394521,male
3,-259.55832,201.61156,51.769340,20.846027,-29.757542,19.461174,6.832013,-27.939716,7.833923,-24.325490,-30.920860,19.854069,-15.415067,777.795990,1106.377053,35.035447,1367.705708,0.040653,95.446381,male
4,-252.93805,197.35612,47.059464,-5.447221,-44.355255,17.521850,10.325183,-27.567768,11.320424,1.321384,-38.533257,8.471567,-20.628464,1006.496444,1246.199155,34.906451,1742.800214,0.057050,112.545026,male


In [ ]:
# change the dtype of gender column
group6['gender'] = group6['gender'].astype("string")

In [ ]:
# change gender label into binary
group6["gender"] = group6["gender"].apply(lambda x: 1 if x == "female" else 0)

# check the unique values to verify
group6['gender'].unique()

array([0, 1])

In [ ]:
# seperate the features into target columns
X1 = group6.drop(columns = "gender")
Y2 = group6['gender']

In [ ]:
print(X1.shape)
print(Y2.shape)

(23, 19)
(23,)


In [ ]:
group6_pred = model.predict(X1)

In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(group6_pred, Y2))

[[9 3]
 [3 8]]


In [ ]:
print(classification_report(group6_pred,Y2))

              precision    recall  f1-score   support

           0       0.75      0.75      0.75        12
           1       0.73      0.73      0.73        11

    accuracy                           0.74        23
   macro avg       0.74      0.74      0.74        23
weighted avg       0.74      0.74      0.74        23



<font size= 5>Observation</font>
- The logistic regression model was found to be 97% accurate on training and test set, which is a good performance in a controlled environment. But when this was tested to the actual dataset of 23 rows it only achieved 74%. This indicates that the model probably cannot be fully extrapolated to real-world data because of variations in the distribution of data, and extremely small sample size. We suggest that more data gathering and model enhancement is required to enhance the reliability using real data.

#<font size= 5><b>KNN (K-Nearest Neighbors)

In [ ]:
# seperate the features into target columns
X = df_new.drop(columns = "gender")
Y = df_new['gender']

# determine the shape of X
print(X.shape)
# determine the shape of Y
print(Y.shape)

(6913, 19)
(6913,)


In [ ]:
# import the necessary libraries for the KNN machine learning mdel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
# create training set for each

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size =0.3, random_state=42)


# verify the shapes of each sets

print(f'train x: {x_train.shape}')
print(f'trian y: {y_train.shape}')
print(f'test x: {x_test.shape}')
print(f'train y: {y_test.shape}')

train x: (4839, 19)
trian y: (4839,)
test x: (2074, 19)
train y: (2074,)


In [ ]:
# using the Standard scaler we can compressed the values
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [ ]:
# create a KNN model with default neighbor which is 5
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(x_train_scaled, y_train)

KNeighborsClassifier()

In [ ]:
# After training the model lets start predicting with test set which is the unseen values of the model
y_pred = knn.predict(x_test_scaled)

print("\nClassification Report:\n", classification_report(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.97      0.98      1070
           1       0.97      1.00      0.98      1004

    accuracy                           0.98      2074
   macro avg       0.98      0.98      0.98      2074
weighted avg       0.98      0.98      0.98      2074



<font size =5>It got 98% in accuracy lets check the confusion matrix

In [ ]:
print(confusion_matrix(y_test, y_pred))

[[1034   36]
 [   4 1000]]


In [ ]:
# lets try with cross validation
from sklearn.model_selection import cross_val_score

scores = cross_val_score(knn, x_train, y_train, cv=5)
print("Cross-validation scores:", scores)
print("Mean CV accuracy:", scores.mean())

Cross-validation scores: [0.87396694 0.875      0.8822314  0.88946281 0.89555326]
Mean CV accuracy: 0.8832428829044418


<font size = 5> To check the validity of our model lets try with our groups dataset

In [ ]:
# import the created dataset

group6_knn = pd.read_csv("/content/gender_voice_features_GROUP6.csv")
group6_knn.head()
# change the dtype of gender column
group6_knn['gender'] = group6_knn['gender'].astype("string")
# change gender label into binary
group6["gender"] = group6["gender"].apply(lambda x: 1 if x == "female" else 0)

# check the unique values to verify
group6['gender'].unique()

# seperate the features into target columns
X1 = group6.drop(columns = "gender")
Y2 = group6['gender']

print(X1.shape)
print(Y2.shape)

(23, 19)
(23,)


In [ ]:
# since we do a standard scaling in our dataset lets do the same with groups dataset
x_test_scaled_group6 = scaler.transform(X1)

In [ ]:
# after scaling the dataset we predict it using the knn model
group6_pred_knn = knn.predict(x_test_scaled_group6)

In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(Y2, group6_pred_knn))

print(classification_report(Y2, group6_pred_knn))

[[ 5 18]
 [ 0  0]]
              precision    recall  f1-score   support

           0       1.00      0.22      0.36        23
           1       0.00      0.00      0.00         0

    accuracy                           0.22        23
   macro avg       0.50      0.11      0.18        23
weighted avg       1.00      0.22      0.36        23



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


<font size= 5>Observation</font>
- The KNN model recorded 97-99 percentage of accuracy on both the training set and the test set, reflecting a good performance in the evaluation. But on the actual data of 23 rows the accuracy was cut by half to 30 percent. In the confusion matrix, it can be seen that the model only predicted class 0 and was unable to identify class 1, which means that the model had a precision, recall, and F1-score of zero. This indicates that KNN is not generalizing sufficiently to real data, probably because the size of the sample is small and the data distributions vary. More data sampling and parameter optimization should be done to enhance its performance on actual data. It can also mean that it is not possible to use KNN in our dataset, probably because the sample size (23 rows) is small and the classes may be unequal.